# Data Mining Assignment 7 - Izaak King, kingi@rpi.edu
Attached is my submission for assignment 7.

## Data preparation

We first parse our given Breast Cancer Wisconsin (Diagnostic) dataset into a data matrix. ID variable isn't used, and Diagnosis variable will be used as class labels for the classification task. The remaining data matrix, 569 x 30 will be scaled with sklearn's MinMaxScalar.

We code the malign class as +1 and and the benign class as 0. Then, we **augment** our dataset by using a column of ones, to avoid the redundant process in our algorithms and evaluation. Finally, we use klearn.train_test_split, with a random_state of 42, and shuffle = True to create test data of size 100, using the remaining data as training data.

In [38]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from scipy.special import softmax
from sklearn.metrics import accuracy_score, f1_score


rawData = pd.read_csv("wdbc.data", header=None)          # Use Pandas Library to parse inital data file, store in Data frame
diagnosisData = (rawData.iloc[ : , 1]).to_numpy()          #The Second column is the Diagosis Variable
dataMatrix = (rawData.iloc[ : , 2: ]).to_numpy()           # Columns 3-32 are saved and added created into a Data Matrix using NumPy. Column 1, Patient ID, is disregarded.
scaler = MinMaxScaler(feature_range=(0, 1))                # Use sklearn's MinMaxScaler
scaledDataMatrix = scaler.fit_transform(dataMatrix)

labels = np.where(diagnosisData == 'M', 1, 0)        # Encode labels with M = 1, B = 0

n = scaledDataMatrix.shape[0]
scaledDataMatrix = np.hstack([np.ones((n,1)), scaledDataMatrix]) 
X_train, X_test, y_train, y_test = train_test_split(
    scaledDataMatrix,
    labels,
    test_size=100,
    random_state=42,
    shuffle=True
)

## QR Factorization

Now, we implement the QR factorization, as described in Section 23.3.1 of the textbook. This function will be used in our Linear Regression Algorithm. 

In [39]:
def qr_factorization(D):
    """
    QR factorization, as found in the textbook
    Input:
        D: n × d matrix
    Returns:
        Q = matrix with orthogonal columns
        R = upper triangular d × d matrix
    """
    n, d = D.shape
    Q = np.zeros((n, d))
    R = np.zeros((d, d))

    for j in range(d):
        v = D[:, j].copy()

        # Compute projections onto earlier U_i
        for i in range(j):
            R[i, j] = np.dot(Q[:, i], D[:, j]) / np.dot(Q[:, i], Q[:, i])
            v -= R[i, j] * Q[:, i]

        # Assign U_j
        Q[:, j] = v
        
        # Unit diagonal, as in Algorithm 23.1
        R[j, j] = 1.0

    return Q, R

We test this function by performing the QR Factorization on our training dataset, ensuring that factors combine to be the original test dataset. In this case, we subtract D - QR, getting a value near 0, indicating that D ≈ QR.

In [44]:
Q, R = qr_factorization(X_train)
# Compute reconstruction of X_train
reconstructed = Q @ R

# Compute the Frobenius norm of the difference
error = np.linalg.norm(reconstructed - X_train, ord='fro')

print("||Q R - X_train|| =", error)



||Q R - X_train||_F = 6.550443575590851e-15


## Linear Regression

Next, we implement the linear regression algorithm via QR factorization, as found in Algorithm 23.1 of the textbook. A helper function was created to solve the back-substitution step in linear regression.

In [40]:
def backsubstitution(R, b):
    """
    Solve R x = b for upper-triangular R using back-substitution
    """
    d = R.shape[0]
    x = np.zeros(d)
    for i in reversed(range(d)):
        x[i] = b[i]
        for j in range(i+1, d):
            x[i] -= R[i, j] * x[j]
        x[i] /= R[i, i]  
    return x


def linear_regression(X, Y):
    """
    Linear regression via QR factorization, as found in the textbook
    Inputs:
        X: n x d data matrix (without bias column)
        Y: n response vector
    Returns:
        w: weights vector (augmented, including bias)
        Y_hat: predicted response vector
    """
    n = X.shape[0]
    
    # QR factorization
    Q, R = qr_factorization(X)

    # Compute delta (squared norms of Q columns)
    delta = np.sum(Q**2, axis=0)
    delta_inv = 1.0 / delta

    # Compute Δ⁻¹ Q^T Y
    Rw = delta_inv * (Q.T @ Y)

    # Solve for w via back substitution
    w = backsubstitution(R, Rw)

    # predicted response
    return w


## Linear Regression Outputs

Now, we use the linear regression function on our training data, outputting the computed weight vector, SSE and R^2 values. We get SSE ≈ 6.60 and R^2 ≈ 0.717, indicating that the linear regression model explains roughly 71.7% of the variance in the training data, with a moderate residual error (SSE ≈ 6.60). This suggests the model fits the training set reasonably, though there is room for improvement.

In [41]:
w = linear_regression(X_train, y_train)

# Print the weight vector
print("Weight vector w:")
print(w)

Y_hat = X_test @ w

# Compute SSE (Sum of Squared Errors)
SSE = np.sum((y_test - Y_hat)**2)
print("\nSSE:", SSE)

# Compute TSS (Total Sum of Squares)
Y_mean = np.mean(y_test)
TSS = np.sum((y_test - Y_mean)**2)

# Compute R^2
R2 = 1 - (SSE / TSS)
print("R^2:", R2)

Weight vector w:
[-0.72795326 -3.94661119  0.12728225  3.47209482  0.32060254 -0.06632257
 -1.65604841  0.38292159  1.07986842 -0.02951732  0.18307457  1.04929698
 -0.08561727  0.03746729 -0.68451153  0.52941726 -0.30660878 -1.57783975
  0.91260638 -0.06767305 -0.04859285  5.89582421  0.30388882 -2.048416
 -3.42527641  0.06462529  0.81116511  0.7152644  -0.3960581   0.52766957
  0.28695959]

SSE: 6.602272820051056
R^2: 0.7167622127820225


## Multiclass Logistic Regression

Next, we implement the multiclass logistic regression algorithm via stochastic gradient ascent, as found in Algorithm 24.2 of the textbook. We also introduce a max_iterations input, which will be used to cap the iterations of this algorithm, in addition to the standard tolerance variable.

In [42]:
def logistic_regression_multiclass(dataset, learning_rate=0.01, tolerance=1e-6, max_iterations=1000):
    """
    Multiclass logistic regression via stochastic gradient ascent (Algorithm 24.2).
    
    Inputs:
        dataset: tuple (x, y)
            x: n x d data matrix (without bias column)
            y: n-dimensional vector of class labels (0..K-1)
        learning_rate: step size for gradient ascent (η)
        tolerance: convergence threshold (ǫ)
        max_iterations: maximum iterations to run
    
    Returns:
        weights: (d) x K matrix of learned weight vectors for each class
        predicted_probs: n x K matrix of predicted probabilities for each sample
    """
    x, y = dataset
    n, d = x.shape
        
    # Number of classes
    num_classes = len(np.unique(y))
    
    # One-hot encoding
    Y_onehot = np.zeros((n, num_classes))
    Y_onehot[np.arange(n), y] = 1
    
    # Random initialization of weights
    weights = np.random.randn(d, num_classes)
    
    for iteration in range(max_iterations):
        weights_old = weights.copy()
        indices = np.random.permutation(n)
        
        for i in indices:
            xi = x[i, :]
            yi = Y_onehot[i, :]
            pi = softmax(xi @ weights)
            
            # Fixed gradient using outer product
            grad = np.outer(xi, (yi - pi))  # (d+1) x K
            weights += learning_rate * grad
        
        # Convergence check
        delta = np.sum(np.abs(weights - weights_old))
        if delta < tolerance:
            print(f"Converged at iteration {iteration}")
            break
    
    predicted_probs = softmax(x @ weights, axis=1)
    return weights, predicted_probs

## Multiclass Logistic Regression Outputs

Now, we use the logistic regression function on our dataset, first using the separated training data to get the weight vectors and probabilities, outputting the accuracy and F1-score for the training data. Next, we evaluate the test set, using scipy.special.softmax. Now we print the accuracy and F1-score of the test data, getting accuracy = 0.97 and F1-score ≈ 0.959, indicating that the logistic regression model achieves highly reliable classification performance on unseen data, correctly identifying nearly all malignant and benign cases.

This result shows that the learned weight vectors generalize well beyond the training set, capturing the discriminative structure of the breast cancer dataset and yielding strong predictive power.

In [43]:
# Run logistic regression on training data
weights, predicted_probs_train = logistic_regression_multiclass(
    (X_train, y_train), learning_rate=0.05, tolerance = 1e-6, max_iterations=5000
)

# Predicted class labels
predicted_labels_train = np.argmax(predicted_probs_train, axis=1)

# Print learned weights
print("Learned weight vectors:\n", weights)


# Evaluate on test set
predicted_probs_test = softmax(
    X_test @ weights, axis=1
)
predicted_labels_test = np.argmax(predicted_probs_test, axis=1)

accuracy_test = accuracy_score(y_test, predicted_labels_test)
f1_test = f1_score(y_test, predicted_labels_test)

print("\nTest Accuracy:", accuracy_test)
print("Test F1-score:", f1_test)

Learned weight vectors:
 [[ 14.89231649 -14.31634218]
 [  1.41705967  -3.02878308]
 [ -0.21237123   0.57049449]
 [  1.36786727  -2.43879382]
 [ -0.95642109   1.4222429 ]
 [  1.30408867   1.20451332]
 [ 13.39046535 -12.31344198]
 [ -1.13267836   0.66474247]
 [-13.55742807  14.56596189]
 [  3.45560992  -2.75253008]
 [ -3.74211657   3.29002504]
 [-16.55125404  17.85782488]
 [  2.6120826   -3.30308617]
 [ -7.03942714   5.43334042]
 [-10.28325747   8.28241195]
 [ -4.31864966   4.49727712]
 [  1.03327277  -2.92653467]
 [  8.65671593  -8.27274527]
 [ -4.97230218   3.87535443]
 [  5.18134287  -6.0840703 ]
 [  6.50934828  -5.24376593]
 [-10.13374093   8.70579732]
 [ -8.51756763   9.44000111]
 [ -0.19591278   0.12171222]
 [ -9.34136052   7.86192309]
 [ -1.63926795   0.11361491]
 [ -0.84228886   0.06017982]
 [ -8.57236974   9.84195029]
 [ -1.27233035   3.73781326]
 [-13.7165413   12.47239701]
 [  1.15428895  -1.14594177]]

Test Accuracy: 0.98
Test F1-score: 0.972972972972973
